# Qwen TTS PPTX on Colab (Gradio)

Run only these cells in order.

1. Set Runtime -> Change runtime type -> GPU (T4).
2. Run Cell 1 (setup + clone + install).
3. Run Cell 2 (launch Gradio).
4. Open the public Gradio URL and upload files in the UI.

Required in UI:
- PPTX
- Ref Audio (wav)
- Ref Text (txt)
- For Colab video build: `slides.zip` with `page1.png`, `page2.png`, ...


In [ ]:
# Cell 1: Setup (idempotent, absolute paths)
REPO_URL = "https://github.com/mkt-kuno/qwen_tts_pptx"
REPO_DIR = "/content/qwen_tts_pptx"

!nvidia-smi
!apt-get -y update
!apt-get -y install ffmpeg sox
!python -m pip install -U pip
!python -m pip install --extra-index-url https://download.pytorch.org/whl/cu124 \
  "cffi>=1.17" "torch==2.9.1" "torchaudio==2.9.1" "torchvision==0.24.1" \
  "gradio>=5,<6" "qwen-tts==0.1.1"

import os, shutil
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone {REPO_URL} {REPO_DIR}
%cd /content/qwen_tts_pptx

import torch
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
from qwen_tts import Qwen3TTSModel
print("qwen_tts import ok")

In [ ]:
# Cell 2: Launch Gradio (Colab mode)
%cd /content/qwen_tts_pptx
import importlib
mod = importlib.import_module("app.web.gradio_app")
if hasattr(mod, "launch_for_colab"):
    mod.launch_for_colab()
else:
    build_app = getattr(mod, "build_app")
    try:
        app = build_app(default_device="cuda:0", default_dtype="float16")
    except TypeError:
        app = build_app()
    app.queue(default_concurrency_limit=1)
    app.launch(share=True, debug=True)
